# import needed libraries
from langchain as well as pysrt, os, uuid

To run, fill in filepath, chunk size, overlap, openai api key and collection name below

In [ ]:
from langchain.schema.document import Document
from langchain.chains import RetrievalQA
from langchain.document_loaders import TextLoader
from langchain.document_loaders import SRTLoader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.llms import LlamaCpp
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.llms import LlamaCpp
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.chains.query_constructor.schema import AttributeInfo
from langchain.document_loaders.merge import MergedDataLoader
from langchain.retrievers import ParentDocumentRetriever
from langchain.vectorstores import Chroma
from langchain.embeddings import GPT4AllEmbeddings
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore
from langchain.vectorstores import PGVector
from langchain.embeddings.openai import OpenAIEmbeddings
import pysrt
import os
import uuid

# adjusted function to spilt a file into langchain documents
partially taken from [this Stackoverflow question](https://stackoverflow.com/questions/48381870/a-better-way-to-split-a-sequence-in-chunks-with-overlaps),
loops over characters of a file and returns langchain documents of size with overlap with metadata (takes the very first timestamp of a chunk)

In [ ]:
def gen_split_overlap(subs, size, overlap, filesource):
    seq = subs.text.replace('\n','')
    documents = []
    position = 0
    current_sub = subs[0]
    if size < 1 or overlap < 0:
        raise ValueError('size must be >= 1 and overlap >= 0')

    for i in range(0, len(seq) - overlap, size - overlap):
        total_len = 0
        for sub in subs:
            if (total_len <= i):
                total_len = total_len + len(sub.text)
                current_sub = sub
        timestamp = current_sub.start
        page = Document(page_content=seq[i:i + size],
        metadata={ "source": filesource,
                  "timestamp": str(timestamp),
                 "uuid": uuid.uuid5(uuid.NAMESPACE_DNS, filesource+str(timestamp))})
        documents.append(page)
    return documents

# loop over all files
(SET FILEPATH AND CHUNK SIZE / OVERLAP HERE)

In [ ]:
# set filpath to directory where your srt files are (without "/" at the end)
filepath = ""
# set chunk size and overlap as you think it is best
chunk_size=500
chunk_overlap=100
from os import listdir
from os.path import isfile, join
files = [f for f in listdir(filepath) if isfile(join(filepath, f))]
documents = []
for file in files:   
    subs = pysrt.open(filepath + "/" + file)
    for sub in subs:
        sub.text = sub.text + " "
    

    documents.extend(gen_split_overlap(subs, chunk_size, chunk_overlap, file))

# print the first documents
for demonstration purposes

In [ ]:
for document in documents[:4]:
    print(document)

# creates embeddings and writes into Vector DB
ADD OPENAI API KEY IF YOU ARE SURE YOU WANT TO WRITE INTO THE VECTOR DB, GIVE COLLECTION A NEW NAME

In [ ]:
# your openai api key
os.environ["OPENAI_API_KEY"]=""
# name of collection, should be distinct from existing ones if you don't want to risk overwrite or adding
COLLECTION_NAME = ""

connection_string = PGVector.connection_string_from_db_params(
     driver=os.environ.get("PGVECTOR_DRIVER", "psycopg2"),
     host=os.environ.get("PGVECTOR_HOST", "185.216.179.150"),
     port=int(os.environ.get("PGVECTOR_PORT", "33101")),
     database=os.environ.get("PGVECTOR_DATABASE", "vectordb"),
     user=os.environ.get("PGVECTOR_USER", "root"),
     password=os.environ.get("PGVECTOR_PASSWORD", "qzx5vQG9WQ2b35eZUWujPUhVb8xRr"),
 )

CONNECTION_STRING = connection_string
embeddings = OpenAIEmbeddings()
vectorestore = PGVector.from_documents(
                embedding=embeddings,
                documents=documents,
                collection_name=COLLECTION_NAME,
                connection_string=CONNECTION_STRING,
            )